# Arrays

In [ ]:
from pathlib import Path
import yaml
from metasmith.python_api import *
from metasmith import examples

In [ ]:
dtypes, contigs, references, transforms = examples.GenomicsAnnotation()
for x in [dtypes, contigs, references, transforms]:
    print(type(x))

In [ ]:
# show contents of input XGDB
for x, d in contigs.manifest.items():
    s = 20-len(str(x))
    print(f"{x}{' '*s}({d})")

In [ ]:
# show contents of reference XGDB (for databases, tool containers, other data dependencies, etc.)
for x, d in references.manifest.items():
    s = 20-len(str(x))
    print(f"{x}{' '*s}({d})")

In [ ]:
# show contents of transforms (available bioinformatics tools)
for x in transforms.manifest:
    print(x.stem)

In [ ]:
path_to_agent_home = Path("./metasmith_home").resolve()
agent = Agent(
    home = Source.FromLocal(path_to_agent_home),
)
if not path_to_agent_home.exists():
    agent.Deploy()
else:
    print("already deployed")

In [ ]:
# suppose we had 3 samples of the same type
# and we want to process them in parallel
SAMPLES = [f"{i+1:02}" for i in range(3)]

# we first create a new xgdb (DataInstanceLibrary) to hold the input array
path_to_array_xgdb = path_to_agent_home/"data/temp/contig_array.xgdb"
input_array = DataInstanceLibrary(path_to_array_xgdb)
# add the original data types to the new xgdb
for namespace, lib in contigs.types.items():
    input_array.AddTypeLibrary(namespace, lib)

# assuming the sample names are unique
# we can use the name as an additional property to specify each sample
# this way, each input is still a contig, but more specifically, the contigs for sample n
contig_properties = dtypes.types["contigs"].properties
samples = DataTypeLibrary(
    types = {sample_name: Endpoint(properties={sample_name}|contig_properties) for sample_name in SAMPLES},
)

# this registers the data type for each sample into the xgdb
input_array.AddTypeLibrary("sample", samples)

print("the original contig type")
print(yaml.safe_dump(dtypes["contigs"].Pack()))
print("sample 01")
print(yaml.safe_dump(samples["01"].Pack()))

In [ ]:
# we can now add each sample to the new data instance library
# by specifying the data type for each sample that includes the sample name
# for example "sample::01"
# since we don't have 3 different samples for this example,
# we can just copy the same contig file 3 times
template = contigs.location/"fosmid.fna"
input_array.Add(
    items = [
        (template, f"{template.stem}_{sample_name}.fna", f"sample::{sample_name}")
        for sample_name in SAMPLES
    ],
)

# the same as in the tutorial https://metasmith.readthedocs.io/en/latest/main/workflow.html
# we can give metasmith the new input_array xgdb
# and ask it to create annotations for each sample using the lineage constraint
task = agent.GenerateWorkflow(
    given=[input_array, references],
    transforms=[transforms],
    targets=[
        dtypes["orf_annotations"].WithLineage([samples[sample_name]]) # we want annotations for each sample
        for sample_name in SAMPLES
    ]
)

# this show the workflow plan
for step in task.plan.steps:
    print(f"step {step.order}: {step.transform.name}:{step.transform.model.key}")
    print(f"    uses: {[str(x.path)+':'+x.dtype.key for x in step.uses]}")
    print(f"   makes: {[str(x.path)+':'+x.dtype.key for x in step.produces]}")
    print()

In [ ]:
agent.StageWorkflow(task, on_exist="clear")
# agent.StageWorkflow(task, on_exist="update")

In [ ]:
agent.RunWorkflow(task)

In [ ]:
agent.CheckWorkflow(task)

In [ ]:
# the printout above should show the path to results
results_path = agent.home.GetPath()/f"runs/{task.plan._key}/results"
results = DataInstanceLibrary.Load(results_path)

# show the results
for path, dtype_name, dtype in results.Iterate():
    parents = results.parents.get(path, [])
    print(f"{dtype_name} at <results>/{path} was made from {[p.name for p in parents]}")